# 새 원본 센서 CSV 추론
학습 없이 저장 모델을 읽고 피처 생성 → 가스 추론. 미성형 미연결을 정상으로 표시하지 않는다.

In [1]:
from pathlib import Path
import nbformat, pandas as pd
root=Path.cwd()
if not (root/'models').exists():root=root.parent
# Only helper definitions are loaded. Export/training cell is NEVER executed.
definitions=nbformat.read(root/'notebooks/154_rg3_gas_model_export.ipynb',as_version=4).cells[1].source
exec(compile(definitions,'rg3_inference_helpers','exec'))
bundle=load_bundle(root/'models/rg3_gas_final_fit.joblib')
# Supply your NEW raw sensor file and optional earlier equipment history here.
# No PassOrFail, Reason, model fitting, or saved evaluation prediction CSV is required.
INPUT_CSV=root/'examples/rg3_gas_raw_input.csv'
CONTEXT_CSV=root/'examples/rg3_gas_raw_history.csv'
OUTPUT_CSV=root/'reports/tables/rg3_new_input_inference.csv'
if INPUT_CSV is None:
 print('Set INPUT_CSV to a raw CSV with EQUIP_CD, product(or PART_NAME), TimeStamp and the 24 raw sensors.')
 print('Provide earlier equipment history through CONTEXT_CSV for past10 features. Standardized KAMP values are NOT raw input.')
else:
 query_raw=pd.read_csv(INPUT_CSV)
 context_raw=pd.read_csv(CONTEXT_CSV) if CONTEXT_CSV else None
 features=prepare_features(query_raw,context_raw)
 prediction=predict_cascade(bundle,features)
 OUTPUT_CSV.parent.mkdir(exist_ok=True,parents=True);prediction.to_csv(OUTPUT_CSV,index=False,encoding='utf-8-sig')
 print(prediction.predicted_type.value_counts().to_string())
 print('Saved',OUTPUT_CSV)
 print('Gas negatives are NOT declared normal while the original shortshot model remains unavailable.')


predicted_type
가스 후보    5
Saved C:\Users\hayeon\workspace\KAMP-IMM\reports\tables\rg3_new_input_inference.csv
Gas negatives are NOT declared normal while the original shortshot model remains unavailable.
